In [10]:

def evaluate(expr, model):
    if isinstance(expr, str):
        return model[expr]

    op = expr[0]

    if op == "NOT":
        return not evaluate(expr[1], model)
    if op == "AND":
        return evaluate(expr[1], model) and evaluate(expr[2], model)
    if op == "OR":
        return evaluate(expr[1], model) or evaluate(expr[2], model)
    if op == "IMPLIES":
        return not evaluate(expr[1], model) or evaluate(expr[2], model)


def get_symbols(expr):
    if isinstance(expr, str):
        return {expr}
    return set().union(*(get_symbols(x) for x in expr[1:]))


def tt_entails(kb, query):
    symbols = set(get_symbols(query))
    for sentence in kb:
        symbols.update(get_symbols(sentence))

    return tt_check_all(kb, query, sorted(symbols), {})


def tt_check_all(kb, query, symbols, model):
    if not symbols:
        if all(evaluate(s, model) for s in kb):
            return evaluate(query, model)
        return True

    p = symbols[0]
    rest = symbols[1:]

    return (
        tt_check_all(kb, query, rest, {**model, p: True})
        and
        tt_check_all(kb, query, rest, {**model, p: False})
    )


kb = [
    ("IMPLIES", "Q", "P"),
    ("IMPLIES", "P", ("NOT", "Q")),
    ("OR", "Q", "R")
]

queries = [
    ("R", "R"),
    ("R → P", ("IMPLIES", "R", "P")),
    ("Q → R", ("IMPLIES", "Q", "R"))
]

print("\n" + "=" * 42)
print("    PROPOSITIONAL LOGIC ENTAILMENT")
print("=" * 42)

print("\nKNOWLEDGE BASE:")
print("  1. Q → P")
print("  2. P → ¬Q")
print("  3. Q ∨ R")

print("\n" + "-" * 42)

for name, query in queries:
    result = tt_entails(kb, query)
    print(f"\n  Query : KB ⊨ {name}")
    print(f"  Result: {'ENTAILED' if result else 'NOT ENTAILED'}")

print("\n" + "=" * 42)



    PROPOSITIONAL LOGIC ENTAILMENT

KNOWLEDGE BASE:
  1. Q → P
  2. P → ¬Q
  3. Q ∨ R

------------------------------------------

  Query : KB ⊨ R
  Result: ENTAILED

  Query : KB ⊨ R → P
  Result: NOT ENTAILED

  Query : KB ⊨ Q → R
  Result: ENTAILED

